# T2.3 — Unit Mapping for Numeric Attributes

This notebook maps every numeric attribute in the `collisions` table to a unit-of-measurement URI from an ontology, and writes the mapping back to DBRepo via the REST API.

**Ontology priority (from the PDF):**
1. SI Digital Framework
2. OMG Commons "Quantities and Units" (fallback)
3. Other ontology — with justification

We use SI for 4 of the 9 mappings (metre, degree). The other 5 go to QUDT, because:
- SI doesn't include imperial units (mph) or non-SI time units (year)
- OMG Commons is a meta-ontology — it defines unit *classes* but no concrete unit *instances*
- QUDT is referenced by OMG Commons itself as the de-facto unit ontology

In [2]:
%pip install -q requests

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup and Authentication

Connects to the DBRepo REST API using HTTP Basic Auth. Username and password are entered at runtime, never stored in the file. The cell ends with a test GET to verify everything works before continuing.

In [3]:
import requests
import json
import os
from getpass import getpass

DBREPO_BASE_URL = "https://test.dbrepo.tuwien.ac.at"
API_BASE = f"{DBREPO_BASE_URL}/api/v1"
DATABASE_ID = "82c19b39-246c-4409-b25c-8baf3a158a70"
COLLISIONS_TABLE_ID = "a5b0155a-9fc6-48f6-8b24-1f591dbf5cad"

USERNAME = input("DBRepo username: ")
PASSWORD = getpass("DBRepo password: ")

AUTH = (USERNAME, PASSWORD)

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
}

test = requests.get(
    f"{API_BASE}/database/{DATABASE_ID}/table/{COLLISIONS_TABLE_ID}",
    auth=AUTH,
    headers=HEADERS,
)
print(f"Test request status: {test.status_code}")
if test.ok:
    print("Authentication OK.")
else:
    print(f"Body: {test.text[:500]}")

Test request status: 200
Authentication OK.


In [4]:
table_url = f"{API_BASE}/database/{DATABASE_ID}/table/{COLLISIONS_TABLE_ID}"
resp = requests.get(table_url, auth=AUTH, headers=HEADERS)
resp.raise_for_status()
table_meta = resp.json()

print(f"Table: {table_meta.get('internal_name')}")
print(f"Columns: {len(table_meta.get('columns', []))}\n")
for col in table_meta["columns"]:
    print(f"  - {col['internal_name']:40s}  type={col.get('column_type', col.get('type'))}")

Table: collisions
Columns: 39

  - collision_index                           type=varchar
  - collision_year                            type=int
  - collision_ref_no                          type=varchar
  - location_easting_osgr                     type=int
  - location_northing_osgr                    type=int
  - longitude                                 type=float
  - latitude                                  type=float
  - lsoa_of_accident_location                 type=varchar
  - number_of_vehicles                        type=int
  - number_of_casualties                      type=int
  - date                                      type=varchar
  - time                                      type=varchar
  - day_of_week                               type=int
  - collision_severity                        type=int
  - enhanced_severity_collision               type=int
  - collision_injury_based                    type=int
  - collision_adjusted_severity_serious       type=float
  - firs

## 3. Unit Mappings

The mappings below cover every numeric attribute in `collisions` that represents an actual physical measurement.

**Not included** (and the reason):
- FK columns like `weather_conditions`, `light_conditions`, `road_type`, `police_force`, `day_of_week`, … — these are categorical codes, not measurements
- `first_road_number`, `second_road_number` — road IDs, not measurements
- `lsoa_of_accident_location` — UK Census geographic code (identifier)
- `collision_index`, `collision_ref_no` — identifiers
- `enhanced_severity_collision`, `collision_injury_based` — derived categorical flags
- `date`, `time` — stored as text in the source

In [5]:
unit_mappings = {
    "location_easting_osgr": {
        "uri": "https://si-digital-framework.org/SI/units/metre",
        "label": "metre",
        "ontology": "SI Digital Framework",
        "justification": (
            "British National Grid easting is expressed in metres, directly "
            "mappable to the SI base unit for length."
        ),
    },
    "location_northing_osgr": {
        "uri": "https://si-digital-framework.org/SI/units/metre",
        "label": "metre",
        "ontology": "SI Digital Framework",
        "justification": (
            "British National Grid northing in metres, SI base unit for length."
        ),
    },
    "longitude": {
        "uri": "https://si-digital-framework.org/SI/units/degree",
        "label": "degree",
        "ontology": "SI Digital Framework",
        "justification": (
            "WGS84 longitude is recorded in decimal degrees. SI explicitly lists "
            "degree as a non-SI unit accepted for use with the SI."
        ),
    },
    "latitude": {
        "uri": "https://si-digital-framework.org/SI/units/degree",
        "label": "degree",
        "ontology": "SI Digital Framework",
        "justification": "WGS84 latitude in decimal degrees, see longitude.",
    },
    "speed_limit": {
        "uri": "https://qudt.org/vocab/unit/MI-PER-HR",
        "label": "mile per hour",
        "ontology": "QUDT",
        "justification": (
            "UK road speed limits are legally defined in miles per hour. "
            "SI does not include imperial units; OMG Commons is a meta-ontology "
            "with no concrete unit instances. QUDT, referenced by OMG Commons "
            "itself as the de-facto unit ontology, provides a stable URI for mph."
        ),
    },
    "collision_year": {
        "uri": "https://qudt.org/vocab/unit/YR",
        "label": "year",
        "ontology": "QUDT",
        "justification": (
            "The SI base unit for time is the second; year is not part of SI. "
            "OMG Commons defines no specific unit instances. QUDT provides a "
            "curated URI for calendar year, suitable for annual aggregation."
        ),
    },
    "number_of_vehicles": {
        "uri": "https://qudt.org/vocab/unit/NUM",
        "label": "number (count)",
        "ontology": "QUDT",
        "justification": (
            "Dimensionless count of vehicles per collision. SI represents "
            "dimensionless quantities as 'one' but does not publish a "
            "browseable URI for this concept. QUDT NUM provides a stable URI."
        ),
    },
    "number_of_casualties": {
        "uri": "https://qudt.org/vocab/unit/NUM",
        "label": "number (count)",
        "ontology": "QUDT",
        "justification": (
            "Dimensionless count of casualties per collision, see number_of_vehicles."
        ),
    },
    "collision_adjusted_severity_serious": {
        "uri": "https://qudt.org/vocab/unit/FRACTION",
        "label": "fraction",
        "ontology": "QUDT",
        "justification": (
            "Probability value in [0, 1] indicating adjusted likelihood that a "
            "collision was of 'serious' severity. QUDT FRACTION captures both "
            "dimensionlessness and the 0-to-1 range characteristic of fractions."
        ),
    },
}

print(f"Total mappings to apply: {len(unit_mappings)}")

Total mappings to apply: 9


## 4. Match Columns to IDs

Maps each column name in `unit_mappings` to its DBRepo column UUID. Raises an error if any of our intended columns doesn't actually exist in the table.

In [6]:
columns_by_name = {col["internal_name"]: col["id"] for col in table_meta["columns"]}

missing = [name for name in unit_mappings if name not in columns_by_name]
if missing:
    raise ValueError(f"These columns are not in the table: {missing}")

print("All mapped columns exist in the table. Column IDs resolved.")

All mapped columns exist in the table. Column IDs resolved.


## 5. Apply Mappings via REST API

Sends a `PUT` to the "Update semantics" endpoint for each column. The payload has three fields — `description`, `concept_uri`, `unit_uri` — and we only touch `unit_uri`; the other two are fetched and passed through again, so we don't overwrite values unintentionally.

In [9]:
def set_column_unit(column_meta: dict, mapping: dict) -> dict:
    column_id = column_meta["id"]
    url = f"{API_BASE}/database/{DATABASE_ID}/table/{COLLISIONS_TABLE_ID}/column/{column_id}"

    existing_description = column_meta.get("description") or ""
    existing_concept = column_meta.get("concept")
    existing_concept_uri = existing_concept.get("uri") if existing_concept else ""

    payload = {
        "description": existing_description,
        "concept_uri": existing_concept_uri,
        "unit_uri": mapping["uri"],
    }

    resp = requests.put(url, auth=AUTH, headers=HEADERS, json=payload)
    return {
        "method": "PUT",
        "url": url,
        "payload": payload,
        "status": resp.status_code,
        "body": resp.text[:500],
    }


columns_by_name_full = {col["internal_name"]: col for col in table_meta["columns"]}

print(f"Applying {len(unit_mappings)} mappings\n")
for col_name, mapping in unit_mappings.items():
    col_meta = columns_by_name_full[col_name]
    result = set_column_unit(col_meta, mapping)
    print(f"[{col_name}]")
    print(f"  {result['method']} {result['url']}")
    print(f"  payload: {json.dumps(result['payload'])}")
    print(f"  -> status {result['status']}")
    if result['status'] >= 400:
        print(f"  body: {result['body']}")
    print()

Applying 9 mappings

[location_easting_osgr]
  PUT https://test.dbrepo.tuwien.ac.at/api/v1/database/82c19b39-246c-4409-b25c-8baf3a158a70/table/a5b0155a-9fc6-48f6-8b24-1f591dbf5cad/column/d4be895c-10ed-467b-847e-7fe912555124
  payload: {"description": "", "concept_uri": "", "unit_uri": "https://si-digital-framework.org/SI/units/metre"}
  -> status 202

[location_northing_osgr]
  PUT https://test.dbrepo.tuwien.ac.at/api/v1/database/82c19b39-246c-4409-b25c-8baf3a158a70/table/a5b0155a-9fc6-48f6-8b24-1f591dbf5cad/column/fd5cba43-49c4-4370-a3f8-75fa6d9b0281
  payload: {"description": "", "concept_uri": "", "unit_uri": "https://si-digital-framework.org/SI/units/metre"}
  -> status 202

[longitude]
  PUT https://test.dbrepo.tuwien.ac.at/api/v1/database/82c19b39-246c-4409-b25c-8baf3a158a70/table/a5b0155a-9fc6-48f6-8b24-1f591dbf5cad/column/baeedc92-e48a-4b8a-acfd-588877bbda9a
  payload: {"description": "", "concept_uri": "", "unit_uri": "https://si-digital-framework.org/SI/units/degree"}
  -> st

## 6. Verify

Re-reads the table schema from DBRepo and prints the unit URI now stored for each mapped column. Mostly a sanity check that the writes landed.

In [7]:
resp = requests.get(table_url, auth=AUTH, headers=HEADERS)
resp.raise_for_status()
updated_meta = resp.json()

print(f"{'Column':<42} {'Unit URI':<60}")
print("-" * 105)
for col in updated_meta["columns"]:
    name = col["internal_name"]
    unit = col.get("unit_uri") or col.get("unit") or col.get("measurement_unit") or "-"
    if name in unit_mappings:
        print(f"{name:<42} {unit:<60}")

Column                                     Unit URI                                                    
---------------------------------------------------------------------------------------------------------
collision_year                             https://qudt.org/vocab/unit/YR                              
location_easting_osgr                      https://si-digital-framework.org/SI/units/metre             
location_northing_osgr                     https://si-digital-framework.org/SI/units/metre             
longitude                                  https://si-digital-framework.org/SI/units/degree            
latitude                                   https://si-digital-framework.org/SI/units/degree            
number_of_vehicles                         https://qudt.org/vocab/unit/NUM                             
number_of_casualties                       https://qudt.org/vocab/unit/NUM                             
collision_adjusted_severity_serious        https://qudt.org/vo

## 7. Write Documentation File

Generates `docs/unit_mapping_table.md` — a readable table of all the mappings plus the reasoning for each one. This is the file we link from the final report.

In [9]:
os.makedirs("../docs", exist_ok=True)

with open("../docs/unit_mapping_table.md", "w", encoding="utf-8") as f:
    f.write("# Unit Mappings — `collisions` Table\n\n")
    f.write("This document records the unit-of-measurement annotations applied to the numeric attributes of the `collisions` table in DBRepo.\n\n")
    f.write("**Selection methodology:** SI Digital Framework was checked first, then OMG Commons. OMG Commons is a meta-ontology with no concrete unit instances, so unmapped attributes were assigned QUDT URIs (referenced by OMG Commons itself as the de-facto unit ontology).\n\n")
    f.write("| Column | Unit | Ontology | URI |\n")
    f.write("|---|---|---|---|\n")
    for col, m in unit_mappings.items():
        f.write(f"| `{col}` | {m['label']} | {m['ontology']} | [{m['uri']}]({m['uri']}) |\n")
    f.write("\n## Justifications\n\n")
    for col, m in unit_mappings.items():
        f.write(f"### `{col}`\n{m['justification']}\n\n")

print("Wrote docs/unit_mapping_table.md")

Wrote docs/unit_mapping_table.md
